In [17]:
import paramiko
import sys

import netCDF4 as nc
import xarray as xr
import wrf

import pandas as pd

In [18]:
def format_ds(wrf_ds):
    lat = wrf_ds['XLAT'].values[:,0]
    lon = wrf_ds['XLONG'].values[0,:]
    time = wrf_ds['Time'].values
    values = wrf_ds.values

    ds = xr.DataArray(
        data=values,
        dims=['time', 'lat', 'lon'],
        coords = {'time' : time, 'lat' : lat, 'lon' : lon}
    )

    return ds

def get_wrf_cld(temp_fpath):
    # Define cloud thresholds
    low_cloud=30
    mid_cloud=2000
    high_cloud=5000

    # Extract cloud fractions and subset to low clouds
    temp_nc = nc.Dataset(temp_fpath)
    cld_frac = wrf.g_cloudfrac.get_cloudfrac(temp_nc,timeidx=wrf.ALL_TIMES,vert_type='height_agl',
                                            low_thresh=low_cloud,
                                            mid_thresh=mid_cloud,
                                            high_thresh=high_cloud) # this is all three layers, we subselect later
    cld_frac = cld_frac.sel(low_mid_high='low')
    temp_nc.close()

    # Subset both datasets to same frame
    lonmin = -119.5049
    lonmax = -119.9401
    latmin = 33.9487
    latmax = 34.1

    # Format the dataarray
    temp_da = format_ds(cld_frac)
    temp_da['time'] = temp_da['time'] - pd.to_timedelta(7, unit='h')  # Convert from UTC to PDT
    temp_da = temp_da.sel(lat=slice(latmin, latmax), lon=slice(lonmax, lonmin))  # Subset to SCI

    return temp_da

# Progress callback function
def progress_callback(transferred, total):
    percent = (transferred / total) * 100
    sys.stdout.write(f"\rDownloading... {percent:.2f}%")
    sys.stdout.flush()

In [19]:
# Get list of remote file paths to download
file_list_fpath = '/Users/patmccornack/Documents/ucsb_fog_project/_repositories/sci-wrf-analysis/data/wrf_low_cloud/file-list.txt'
with open(file_list_fpath, "r") as f:
    remote_fpaths = f.read().splitlines()  # Removes newline characters

In [20]:
temp_fpath = "/Users/patmccornack/Documents/ucsb_fog_project/_repositories/sci-wrf-analysis/data/wrf_low_cloud/temp.nc"
wrf_cld_fpath = "/Users/patmccornack/Documents/ucsb_fog_project/_repositories/sci-wrf-analysis/data/wrf_low_cloud/wrf-low-clouds.nc"

host = "great.eri.ucsb.edu"
port = 22  # Default SSH port
username = "patmccornack"
password = "Feetofclay23!"

i = 0

# Use first file to create wrf_low_clouds
remote_file = remote_fpaths[0]

# Establish SSH connection
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(host, port, username, password)

# Use SFTP to download the file with progress tracking
sftp = ssh.open_sftp()
file_size = sftp.stat(remote_file).st_size  # Get file size
print(f"{i} Downloading {remote_file} ({file_size} bytes)")
sftp.get(remote_file, temp_fpath, callback=progress_callback)
sftp.close()

# Get the cloud fraction and save out
wrf_cld_da = get_wrf_cld(temp_fpath)
wrf_cld_da.to_netcdf(wrf_cld_fpath, mode='w')
sftp.close()
i += 1

# Append the rest of the files
for remote_file in remote_fpaths[1:]:
    # Use SFTP to download the file with progress tracking
    sftp = ssh.open_sftp()
    file_size = sftp.stat(remote_file).st_size  # Get file size
    print(f"{i} Downloading {remote_file} ({file_size} bytes)")
    sftp.get(remote_file, temp_fpath, callback=progress_callback)
    sftp.close()

    # Get the cloud fraction and append
    temp_da = get_wrf_cld(temp_fpath)
    wrf_cld_da = xr.concat([wrf_cld_da, temp_da], dim="time")  # Concatenate over time
    wrf_cld_da.to_netcdf(wrf_cld_fpath, mode='w')
    i += 1

# Close connections
ssh.close()

0 Downloading /home/perseverance-clivac/wrf_sbarbara/wrfout/1996/wrfout_d04_1996-06-22_00:00:00 (9832533352 bytes)
Downloading... 100.00%1 Downloading /home/perseverance-clivac/wrf_sbarbara/wrfout/1996/wrfout_d04_1996-06-23_00:00:00 (9832533352 bytes)
Downloading... 100.00%2 Downloading /home/perseverance-clivac/wrf_sbarbara/wrfout/1996/wrfout_d04_1996-06-24_00:00:00 (9832533352 bytes)
Downloading... 100.00%3 Downloading /home/perseverance-clivac/wrf_sbarbara/wrfout/1996/wrfout_d04_1996-06-25_00:00:00 (9832533352 bytes)
Downloading... 100.00%4 Downloading /home/perseverance-clivac/wrf_sbarbara/wrfout/1996/wrfout_d04_1996-06-26_00:00:00 (9832533352 bytes)
Downloading... 100.00%5 Downloading /home/perseverance-clivac/wrf_sbarbara/wrfout/1996/wrfout_d04_1996-06-27_00:00:00 (9832533352 bytes)
Downloading... 100.00%6 Downloading /home/perseverance-clivac/wrf_sbarbara/wrfout/1996/wrfout_d04_1996-06-28_00:00:00 (9832533352 bytes)
Downloading... 100.00%7 Downloading /home/perseverance-clivac/w

SFTPError: Garbage packet received

In [ ]:
# Check file
xr.open_dataarray(wrf_cld_fpath)

<xarray.DataArray (time: 72, lat: 18, lon: 42)> Size: 218kB
[54432 values with dtype=float32]
Coordinates:
  * lat      (lat) float32 72B 33.95 33.96 33.97 33.98 ... 34.08 34.09 34.1
  * lon      (lon) float32 168B -119.9 -119.9 -119.9 ... -119.5 -119.5 -119.5
  * time     (time) datetime64[ns] 576B 2000-04-30T17:00:00 ... 2000-05-01T16...